# Modelling and comparison

Three models × two feature sets × 11 targets.

The loss is fixed at quantile q=0.5 and the split at five-fold grouped by company. 

This stage asks which model family fitsand whether the relative-position feature helps.

**Metrics** 

As EDA 8.3, rank-based metrics only. The discrimination rate and pairwise accuracy form a pair and
must be read together: the first is how often the model commits to an order, the second how often it
is right when it does. Fields with heavy zero-inflation show a low discrimination rate, and accuracy
alone would overstate their usefulness.

**B0**

The B0 baseline predicts no change throughout, and shows whether the model beats assuming stasis.

In [1]:
# Environment and data

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from lightgbm import LGBMRegressor
import sys

folder_01 = Path.cwd().parent / "01 EDA + Data PreProcessing"

sys.path.append(str(folder_01))

import data_prep as dp

OUT_DIR = Path("model_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["CompanyNumber_norm", "period_t", "period_t_plus_1"]
DATE_COLS = ["period_t", "period_t_plus_1", "available_date_t", "available_date_t_plus_1"]

N_SPLITS = 5
MIN_ROWS = 200
RANDOM_STATE = 0
N_PAIRS = 2_000_000     # pairs sampled for the discrimination rate

ALPHAS = np.logspace(-2, 4, 13)

## 1. Load and clean

In [2]:
# Load and clean

pairs = pd.read_csv("../01 EDA + Data PreProcessing/04_Five_CSV/03_financial_change_labels.csv",
                    dtype={"CompanyNumber_norm": str}, low_memory=False)
for c in DATE_COLS:
    pairs[c] = pd.to_datetime(pairs[c], errors="coerce")

meta = pd.read_csv("../01 EDA + Data PreProcessing/01_CompaniesSelected/UKcompanies_active_account_category_sample_100k.csv",
                   dtype={"CompanyNumber": str}, low_memory=False)
meta["CompanyNumber_norm"] = meta["CompanyNumber"].map(dp.normalise_company_number)
meta["IncorporationDate"] = pd.to_datetime(meta["IncorporationDate"], errors="coerce")

raw = pairs.merge(
    meta[["CompanyNumber_norm", "IncorporationDate", "CompanyCategory", "multi_sic_company"]],
    on="CompanyNumber_norm", how="left", validate="many_to_one")

df, _ = dp.clean_global(raw)

assignment = pd.read_csv("../01 EDA + Data PreProcessing/eda_output/split_assignment.csv",
                         dtype={"CompanyNumber_norm": str},
                         parse_dates=["period_t", "period_t_plus_1"])
df = df.merge(assignment, on=KEY, how="left", validate="one_to_one")

train = df[df["split_company"] == "train"].copy()
print(f"train: {len(train):,} rows / {train['CompanyNumber_norm'].nunique():,} companies")

train: 65,293 rows / 61,288 companies


## 2. Evaluation

**Discrimination rate** 

Among pairs whose true values differ, how often the model gives different predictions. A low rate
means it returns identical values for many companies, whose percentiles are then undefined.

**Pairwise accuracy**

`(Kendall tau-b + 1) / 2`: accuracy when the model does commit. Computed via `kendalltau` rather than enumeration, since 74k companies imply 2.7 billion pairs. tau-b drops both pairs tied in truth
(unanswerable) and pairs tied in prediction (the model abstaining, which the exclusion silently forgives), so it must be read alongside the discrimination rate.

**Effective accuracy.** `0.5 + Discrimination rate × (Pairwise accuracy − 0.5)`

Counts abstention as no better than chance. Comparable across fields.

In [3]:
def discrimination_rate(y_true, y_pred, n_pairs=N_PAIRS, seed=0):
    """Among answerable pairs, the share where the model commits to an order"""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    i, j = rng.integers(0, n, n_pairs), rng.integers(0, n, n_pairs)
    answerable = (i != j) & (y_true[i] != y_true[j])
    if not answerable.any():
        return np.nan
    return float((y_pred[i[answerable]] != y_pred[j[answerable]]).mean())


def evaluate(y_true, y_pred):
    """Rank-based metrics"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if np.std(y_pred) < 1e-12:          # constant prediction
        return {"discrim_rate": 0.0, "pairwise_acc": np.nan,
                "effective_acc": 0.5, "spearman": np.nan}

    tau = kendalltau(y_true, y_pred, variant="b").statistic
    acc = (tau + 1) / 2 if np.isfinite(tau) else np.nan
    disc = discrimination_rate(y_true, y_pred)
    return {
        "discrim_rate": disc,
        "pairwise_acc": acc,
        "effective_acc": 0.5 + disc * (acc - 0.5) if np.isfinite(acc) else np.nan,
        "spearman": spearmanr(y_true, y_pred).statistic,
    }

## 3. A single run

The feature matrix is rebuilt inside each fold: `build_matrix` fits on the training fold and the returned statistics are applied to the validation fold. Fitting the winsorise cut-offs, group medians or one-hot category sets on the full data would be leakage at the preprocessing level.

In [4]:
def make_model(name):
    """Model factory"""
    if name == "hgb":
        return HistGradientBoostingRegressor(
            loss="quantile", quantile=0.5, max_iter=300,
            early_stopping=False, random_state=RANDOM_STATE)
    if name == "ridge":
        # Scaling is fitted per fold, hence inside the pipeline
        return make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS))
    raise ValueError(name)


def run_one(frame, model_name, target, variant, peer_pct=False, collect_preds=False):
    """
    Returns:
        records: list of dict (one record per fold per model)
        preds_df: DataFrame | None
    """
    elig = frame[f"{target}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig].copy()
    y = sub[f"{target}_signed_log_change"].astype(float)
    sub, y = sub.loc[y.notna()], y.loc[y.notna()]
    if len(sub) < MIN_ROWS:
        return [], None

    groups = sub["CompanyNumber_norm"]
    records, blocks = [], []

    for fold, (tr, te) in enumerate(GroupKFold(n_splits=N_SPLITS).split(sub, y, groups=groups)):
        tr_frame, te_frame = sub.iloc[tr], sub.iloc[te]
        X_tr, cat_cols, stats = dp.build_matrix(tr_frame, variant, peer_pct=peer_pct)
        X_te, _, _ = dp.build_matrix(te_frame, variant, fit_stats=stats, peer_pct=peer_pct)
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        preds = {"B0_zero": np.zeros(len(y_te))}

        model = make_model(model_name)
        if cat_cols:         # trees take them raw
            model.set_params(categorical_features=[X_tr.columns.get_loc(c) for c in cat_cols])
        model.fit(X_tr, y_tr)
        preds[model_name] = model.predict(X_te)

        base = dict(target=target, model_name=model_name, variant=variant,
                    peer_pct=peer_pct, fold=fold,
                    n_train=len(tr), n_test=len(te),
                    zero_frac=float((y_te == 0).mean()))

        if collect_preds:
            block = te_frame[KEY + ["primary_sector", "acct_cat_model"]].copy()
            block["target"] = target
            block["peer_pct"] = peer_pct
            block["fold"] = fold
            block["y_true"] = y_te.to_numpy()
            block["x_t"] = te_frame[f"{target}_t"].to_numpy()
            block["y_pred"] = preds[model_name]
            blocks.append(block)

        for name, p in preds.items():
            records.append({**base, "model": name, **evaluate(y_te, p)})

    return records, (pd.concat(blocks, ignore_index=True) if blocks else None)

In [5]:
def summarise(records):
    res = pd.DataFrame(records)
    num = ["discrim_rate", "pairwise_acc", "effective_acc", "spearman", "zero_frac"]
    out = res.groupby(["target", "model", "variant", "peer_pct"])[num].mean()
    out["n"] = res.groupby(["target", "model", "variant", "peer_pct"])["n_test"].sum()
    out["acc_sd"] = res.groupby(["target", "model", "variant", "peer_pct"])["pairwise_acc"].std()
    return out.round(4).reset_index()

## 4. HistGBM
11 target × 2 feature matrix = 22

In [6]:
hgb_records = []
for target in dp.METRICS:
    for peer in (False, True):
        recs, _ = run_one(train, "hgb", target, variant="gbm", peer_pct=peer)
        hgb_records += recs
    print(f"done: {target}")

hgb = summarise(hgb_records)
hgb.to_csv(OUT_DIR / "hgb_summary.csv", index=False, encoding="utf-8-sig")

done: current_assets
done: fixed_assets
done: creditors_total
done: equity
done: net_assets_liabilities
done: net_current_assets_liabilities
done: cash
done: debtors
done: employees
done: profit_loss
done: total_assets_less_current_liabilities


In [7]:
def compare_table(summary, model_name):
    """Base feature set against the same set plus peer_pct, by field"""
    m = summary[summary["model"] == model_name]
    base = m[~m["peer_pct"]].set_index("target")
    peer = m[m["peer_pct"]].set_index("target")
    b0 = (summary[summary["model"] == "B0_zero"]
          .drop_duplicates("target").set_index("target"))

    out = pd.DataFrame({
        "n": base["n"], "zero_frac": base["zero_frac"],
        "discrim": base["discrim_rate"],
        "acc": base["pairwise_acc"],
        "eff_acc": base["effective_acc"],
        "spearman": base["spearman"],
        "acc_sd": base["acc_sd"],
        "eff_acc_peer": peer["effective_acc"],
    })
    out["peer_gain"] = (out["eff_acc_peer"] - out["eff_acc"]).round(4)
    return out.sort_values("eff_acc", ascending=False).round(4)


tbl = compare_table(hgb, "hgb")
print("effective_acc counts abstention")
tbl

effective_acc counts abstention


,n,zero_frac,discrim,acc,eff_acc,spearman,acc_sd,eff_acc_peer,peer_gain
target,,,,,,,,,
fixed_assets,32203,0.2404,0.9638,0.6047,0.6010,0.2929,0.0053,0.6003,-0.0007
profit_loss,2351,0.0332,1.0000,0.5979,0.5979,0.2864,0.0102,0.5962,-0.0017
creditors_total,59239,0.0656,0.9999,0.5814,0.5814,0.2362,0.0032,0.5825,0.0011
cash,25551,0.0413,1.0000,0.5798,0.5798,0.2325,0.0077,0.5794,-0.0004
total_assets_less_current_liabilities,56348,0.0620,0.9995,0.5698,0.5698,0.1928,0.0048,0.5699,0.0001
net_current_assets_liabilities,59887,0.0549,0.9995,0.5639,0.5639,0.1805,0.0027,0.5632,-0.0007
current_assets,55886,0.0496,0.9997,0.5609,0.5609,0.1769,0.0041,0.5612,0.0003
net_assets_liabilities,53706,0.0611,0.9999,0.5595,0.5595,0.1630,0.0026,0.5588,-0.0007
equity,63033,0.1282,0.9966,0.5554,0.5552,0.1449,0.0053,0.5521,-0.0031


## 5. Ridge

The linear baseline. Its feature content is identical to GBM's: the same eleven fields, metadata,
rows and folds, differing only in representation.

EDA 7.1 found the relationship hump-shaped, which a linear model cannot fit.

In [8]:
ridge_records = []
for target in dp.METRICS:
    for peer in (False, True):
        recs, _ = run_one(train, "ridge", target, variant="ridge", peer_pct=peer)
        ridge_records += recs
    print(f"done: {target}")

ridge = summarise(ridge_records)
ridge.to_csv(OUT_DIR / "ridge_summary.csv", index=False, encoding="utf-8-sig")

done: current_assets
done: fixed_assets
done: creditors_total
done: equity
done: net_assets_liabilities
done: net_current_assets_liabilities
done: cash
done: debtors
done: employees
done: profit_loss
done: total_assets_less_current_liabilities


In [9]:
compare_table(ridge, "ridge")

,n,zero_frac,discrim,acc,eff_acc,spearman,acc_sd,eff_acc_peer,peer_gain
target,,,,,,,,,
profit_loss,2351,0.0332,1.0,0.6018,0.6018,0.2965,0.0158,0.6131,0.0113
employees,61076,0.7331,1.0,0.5927,0.5927,0.2364,0.0055,0.5941,0.0014
cash,25551,0.0413,1.0,0.5815,0.5815,0.2387,0.0068,0.5801,-0.0014
creditors_total,59239,0.0656,1.0,0.5756,0.5756,0.2224,0.0028,0.5720,-0.0036
debtors,21420,0.1122,1.0,0.5655,0.5655,0.1934,0.0040,0.5658,0.0003
current_assets,55886,0.0496,1.0,0.5538,0.5538,0.1584,0.0017,0.5541,0.0003
net_current_assets_liabilities,59887,0.0549,1.0,0.5395,0.5395,0.1211,0.0035,0.5369,-0.0026
total_assets_less_current_liabilities,56348,0.0620,1.0,0.5239,0.5239,0.0749,0.0032,0.5218,-0.0021
fixed_assets,32203,0.2404,1.0,0.5158,0.5158,0.0486,0.0039,0.5335,0.0177


## 6. LightGBM
LightGBM and HistGBM are two implementations of the same algorithm, differing mainly in defaults and
binning.

The feature matrix is shared with HistGBM; only the categorical columns need converting to `category` dtype.

In [10]:
def to_lgbm_categorical(X_tr, X_te, cat_cols):
    # Category levels taken from the training fold
    X_tr, X_te = X_tr.copy(), X_te.copy()
    for c in cat_cols:
        X_tr[c] = X_tr[c].astype("category")
        X_te[c] = pd.Categorical(X_te[c], categories=X_tr[c].cat.categories)
    return X_tr, X_te


def run_lgbm(frame, target, peer_pct=False, collect_preds=False):
    elig = frame[f"{target}_change_eligible"].fillna(False).astype(bool)
    sub = frame.loc[elig].copy()
    y = sub[f"{target}_signed_log_change"].astype(float)
    sub, y = sub.loc[y.notna()], y.loc[y.notna()]
    if len(sub) < MIN_ROWS:
        return [], None

    groups = sub["CompanyNumber_norm"]
    records, blocks = [], []

    for fold, (tr, te) in enumerate(GroupKFold(n_splits=N_SPLITS).split(sub, y, groups=groups)):
        tr_frame, te_frame = sub.iloc[tr], sub.iloc[te]
        X_tr, cat_cols, stats = dp.build_matrix(tr_frame, "gbm", peer_pct=peer_pct)
        X_te, _, _ = dp.build_matrix(te_frame, "gbm", fit_stats=stats, peer_pct=peer_pct)
        X_tr, X_te = to_lgbm_categorical(X_tr, X_te, cat_cols)
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        model = LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=300,
                              verbose=-1, random_state=RANDOM_STATE)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)

        base = dict(target=target, model_name="lgbm", variant="gbm",
                    peer_pct=peer_pct, fold=fold,
                    n_train=len(tr), n_test=len(te),
                    zero_frac=float((y_te == 0).mean()))

        if collect_preds:
            block = te_frame[KEY + ["primary_sector", "acct_cat_model", "acct_cat_raw"]].copy()
            block["target"], block["peer_pct"], block["fold"] = target, peer_pct, fold
            block["y_true"] = y_te.to_numpy()
            block["x_t"] = te_frame[f"{target}_t"].to_numpy()
            block["pred"] = pred
            blocks.append(block)

        records.append({**base, "model": "B0_zero", **evaluate(y_te, np.zeros(len(y_te)))})
        records.append({**base, "model": "lgbm", **evaluate(y_te, pred)})

    return records, (pd.concat(blocks, ignore_index=True) if blocks else None)

In [11]:
lgbm_records = []
for target in dp.METRICS:
    for peer in (False, True):
        recs, _ = run_lgbm(train, target, peer_pct=peer)
        lgbm_records += recs
    print(f"done: {target}")

lgbm = summarise(lgbm_records)
lgbm.to_csv(OUT_DIR / "lgbm_summary.csv", index=False, encoding="utf-8-sig")

done: current_assets
done: fixed_assets
done: creditors_total
done: equity
done: net_assets_liabilities
done: net_current_assets_liabilities
done: cash
done: debtors
done: employees
done: profit_loss
done: total_assets_less_current_liabilities


In [12]:
compare_table(lgbm, "lgbm")

,n,zero_frac,discrim,acc,eff_acc,spearman,acc_sd,eff_acc_peer,peer_gain
target,,,,,,,,,
fixed_assets,32203,0.2404,0.9545,0.6056,0.6008,0.2948,0.0054,0.6024,0.0016
profit_loss,2351,0.0332,1.0000,0.5956,0.5956,0.2800,0.0181,0.5995,0.0039
cash,25551,0.0413,1.0000,0.5810,0.5810,0.2357,0.0079,0.5806,-0.0004
creditors_total,59239,0.0656,0.9999,0.5809,0.5809,0.2345,0.0036,0.5812,0.0003
total_assets_less_current_liabilities,56348,0.0620,0.9995,0.5708,0.5708,0.1955,0.0050,0.5699,-0.0009
current_assets,55886,0.0496,1.0000,0.5651,0.5651,0.1889,0.0028,0.5659,0.0008
net_current_assets_liabilities,59887,0.0549,0.9998,0.5633,0.5633,0.1790,0.0026,0.5624,-0.0009
debtors,21420,0.1122,1.0000,0.5593,0.5593,0.1744,0.0043,0.5572,-0.0021
net_assets_liabilities,53706,0.0611,0.9998,0.5581,0.5581,0.1587,0.0046,0.5597,0.0016


## Interim findings

**The three models diverge sharply on `employees`**

Ridge reaches an effective accuracy of 0.5927 against 0.5239 for HistGBM and 0.5050 for LightGBM, the last being close to chance. 

The cause is the discrimination rate: at 0.2951 and 0.1965 the two GBMs return identical predictions (mostly zero) for a large share of companies and abstain from ordering them, whereas Ridge, whose output is a continuous linear combination, never abstains. With 73.3% of companies showing no change in headcount, the leaf medians under quantile loss collapse to zero. No other field shows this: all three models exceed a discrimination rate of 0.95 elsewhere.

**GBM leads on most fields**

Apart from `employees`, `cash` and `debtors`, both GBMs outperform Ridge on every field, by between
0.006 and 0.085. HistGBM and LightGBM differ by no more than 0.005。

**The ablation shows no gain. relative-position features are dropped**

Across 33 comparisons, only two exceeded the 0.005 threshold (Ridge on `profit_loss` at +0.0113 and `fixed_assets` at +0.0177). The rest fall within ±0.005, several of them negative.

## 7. Summary comparison

**Why ranks rather than averaged metrics.**
The denominators differ by field, since the share of answerable pairs varies with zero-inflation.
Averaging `effective_acc` across fields is therefore not meaningful. But ranks are dimensionless and
unaffected by denominator or sample size.

`rank_sd` matters as much as `mean_rank`: a low mean with a high variance indicates a model that wins on some fields and loses on others rather than a stable choice.

`profit_loss` is excluded from the ranking: with n=2,351 its fold-to-fold standard deviation is three to five times that of the other fields, so its position is largely noise.

In [ ]:
SUMMARIES = {"hgb": hgb, "ridge": ridge, "lgbm": lgbm}
EXCLUDE_FROM_RANK = ["profit_loss"]  # too few rows, unstable


def model_comparison(summaries, peer_pct=False, exclude=EXCLUDE_FROM_RANK):
    """rank models field by field, then average"""
    frames = []
    for name, s in summaries.items():
        m = s[(s["model"] == name) & (s["peer_pct"] == peer_pct)]
        frames.append(m[["target", "model", "effective_acc", "discrim_rate",
                         "spearman", "n", "zero_frac"]])
    long = pd.concat(frames, ignore_index=True)
    ranked = long[~long["target"].isin(exclude)].copy()
    ranked["rank"] = ranked.groupby("target")["effective_acc"].rank(ascending=False)

    out = ranked.groupby("model").agg(
        mean_rank=("rank", "mean"), # the smaller the better
        rank_sd=("rank", "std"), # the smaller the more stable
        n_best=("rank", lambda r: int((r == 1).sum())),
        mean_eff_acc=("effective_acc", "mean"),
    )
    return out.sort_values("mean_rank").round(3), long


ranks, long = model_comparison(SUMMARIES)
print(f"fields ranked: {len(dp.METRICS) - len(EXCLUDE_FROM_RANK)}\n")
ranks

fields ranked: 10



,mean_rank,rank_sd,n_best,mean_eff_acc
model,,,,
hgb,1.8,0.789,4,0.565
lgbm,1.8,0.632,3,0.564
ridge,2.4,0.966,3,0.548


In [14]:
# Field-level detail
models = list(SUMMARIES)
pivot = long.pivot(index="target", columns="model", values="effective_acc")
pivot["best"] = pivot[models].idxmax(axis=1)
pivot["spread"] = (pivot[models].max(axis=1) - pivot[models].min(axis=1)).round(4)
pivot["zero_frac"] = long.groupby("target")["zero_frac"].first()
pivot["n"] = long.groupby("target")["n"].first()

print(pivot.sort_values("spread", ascending=False).round(4).to_string())

model                                     hgb    lgbm   ridge   best  spread  zero_frac      n
target                                                                                        
employees                              0.5239  0.5050  0.5927  ridge  0.0877     0.7331  61076
fixed_assets                           0.6010  0.6008  0.5158    hgb  0.0852     0.2404  32203
total_assets_less_current_liabilities  0.5698  0.5708  0.5239   lgbm  0.0469     0.0620  56348
net_assets_liabilities                 0.5595  0.5581  0.5127    hgb  0.0468     0.0611  53706
equity                                 0.5552  0.5564  0.5147   lgbm  0.0417     0.1282  63033
net_current_assets_liabilities         0.5639  0.5633  0.5395    hgb  0.0244     0.0549  59887
debtors                                0.5540  0.5593  0.5655  ridge  0.0115     0.1122  21420
current_assets                         0.5609  0.5651  0.5538   lgbm  0.0113     0.0496  55886
profit_loss                            0.5979  0.5

### Conclusions

HistGBM and LightGBM both average rank 1.8 and differ by 0.001 in mean effective accuracy. Excluding
`employees`, their largest per-field gap is 0.0053.

This comparison used default hyperparameters. The two GBMs are indistinguishable. The choice is deferred to tuning.

The main model is chosen from the tuned results.

---

**Ridge trails the trees, subject to a known limitation in its feature representation**

Ridge averages rank 2.4 and trails on eight fields by 0.006 to 0.085, while matching or slightly
beating the trees on `cash`, `debtors` and `profit_loss`. This is consistent with the generally non-linear relationships found in EDA 7.1

**The limitation** 

The three additional steps on the linear branch address, respectively, the separation of the positive and negative clusters, the leverage of extreme values on the coefficients(winsorize), and differing slopes across categories (EDA 7.2). None introduces a basis expansion for the non-linearity within a curve.


The result therefore shows that a linear model is insufficient under the present representation.
not that no linear representation could fit the relationship. Testing the latter would require spline or binned bases, which this project does not attempt.

Ridge is not a candidate for delivery, it serves as a reference for how complex the relationship is.
The limitation does not affect the choice of main model(tree)

---

**`employees` requires separate treatment**

The two-part model in `model_optimization`. The performance will be compared with the Ridge figure(0.5927).
